In [62]:
from ast import literal_eval

from nnscaling import aesthetics
from nnscaling.data import (
    DatasetConfig,
    get_dataset_class,
)
from nnscaling.models import MLP, ScaledModel
from nnscaling.utils import compute_model_accuracy, get_device, safetensors_metadata_parser
from nnscaling.visualization import plot_decision_boundary

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
file_path = "/home/nsa325/work/nnscaling/model_saves/mnist_model.safetensors"

In [64]:
# Dataset config
metadata = safetensors_metadata_parser(file_path=file_path)
dataset_config = DatasetConfig(
    dataset_name=metadata["dataset"],
    num_samples=2_000,
    split="test",
    seed=42,
)
DatasetClass = get_dataset_class(name=dataset_config.dataset_name)
dataset = DatasetClass(config=dataset_config)

In [65]:
model = MLP.load_model(file_path, in_features=dataset.in_features).eval()

In [66]:
print(f"Testing Accuracy: {compute_model_accuracy(model.to('cpu'), dataset)}")

if dataset.in_features == 2:
    plot_decision_boundary(
        model,
        model.state_dict(),
        dataset.features,
        dataset.labels.squeeze(),
        labels=[0, 1, 2],
    )

Testing Accuracy: 0.9684000015258789


In [67]:
file_path = "/home/nsa325/work/nnscaling/model_saves/scaling/loc_1_scaled_mnist_model.safetensors"
scaled_model = ScaledModel.load_model(model, file_path).eval()

{'added_layers': '10', 'config': '[256, 128, 64, 32]', 'dataset': 'MNISTDataset', 'layer_start': '1', 'nonlinearity': 'relu', 'out_features': '10', 'scale_nonlinearity': 'relu'}


In [68]:
print(f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu'), dataset)}")

if dataset.in_features == 2:
    plot_decision_boundary(
        scaled_model,
        scaled_model.state_dict(),
        dataset.features,
        dataset.labels.squeeze(),
        labels=[0, 1],
    )

/home/nsa325/work/nnscaling/nnscaling/models/scaler.py:118: UserWarning: The `forward` method calls the `forward_scaled` method. For vanilla forward, call `forward_raw`.
  warnings.warn(


Testing Accuracy: 0.9606999754905701
